# CPGF step-edge example

This is a short example for trying one filter. It makes a `1024 x 1024` image where the left half is black and the right half is white. Then it runs the Circular Polynomial Gradient Filter (CPGF) and plots the two outputs.

Change `radius` and `degree`, rerun the notebook, and compare the outputs. Those two values are the main CPGF settings to experiment with.


First import only what this example needs. `time` is only here so the notebook can print how long the filter took to run.


In [ ]:
import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import ExecutionPath, cpgf_definition, run_filter

Set the CPGF parameters here. A larger `radius` uses a bigger neighborhood around each pixel. A larger `degree` fits a higher-order polynomial. Some combinations may not be valid because there are not enough pixels in the support to fit that polynomial.

`ExecutionPath.SPARSE_OFFSETS` is a good default for CPGF. You can also try `ExecutionPath.SPATIAL_DENSE` to run the same filter with a dense-kernel path.


In [ ]:
radius = 2  # Wider radius samples more pixels around each point.
degree = 2  # Higher degree fits a higher-order polynomial surface.
path = ExecutionPath.SPARSE_OFFSETS  # Try SPATIAL_DENSE to compare paths.

size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

Now build the CPGF definition and run it. The timer measures only the filter run, not the image setup above.


In [ ]:
definition = cpgf_definition(radius=radius, degree=degree)  # Build the CPGF weights.
start = time.perf_counter()
gradient_x, gradient_y = run_filter(
    definition,
    image,
    path=path,  # Use the execution path selected above.
    boundary=definition.default_boundary,  # Use the boundary mode chosen by CPGF.
)
elapsed = time.perf_counter() - start

print(f"{elapsed:.4f} seconds to run CPGF")

The image changes left-to-right, so the main response should appear in `gradient_x`. The `gradient_y` output should be basically zero because the image does not change top-to-bottom.

Both plots use the same grayscale range. That matters here because plotting `gradient_y` by itself can stretch tiny roundoff values until they look like a real edge.


In [ ]:
limit = max(float(gradient_x.abs().max()), float(gradient_y.abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].imshow(gradient_x[0], cmap="gray", vmin=-limit, vmax=limit)
axes[0].set_title("gradient_x")

axes[1].imshow(gradient_y[0], cmap="gray", vmin=-limit, vmax=limit)
axes[1].set_title("gradient_y")

for ax in axes:
    ax.axis("off")

fig.suptitle(f"CPGF radius={radius}, degree={degree}, time={elapsed:.4f} s")
fig.tight_layout()